In [2]:


import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from tqdm.notebook import tqdm
import time
import os


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/lib/python3/dist-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/chriko/.local/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/usr/lib/python3/dist-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/usr/lib/python3/dist-packages/tornado/platform/asyncio.

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



In [3]:
table2 = pd.DataFrame(
    data={
        "flex_elec_gwh": [12, 12.6, 16.4, 16.4, 12.7, 14.1, 16.6, 17],
        "costs_savings_keur": [489, 520, 621, 632, 518, 573, 633, 658],
        "emissions_saved_tco2": [632, 617, 842, 816, 670, 650, 846, 811],
        "flh_w_flx_h": [7008, 7008, 3711, 3657, 7008, 7008, 3668, 3570],
        "daily_min_capture_%": [66.0, 81.0, 66.0, 81.0, 71.1, 93.0, 71.1, 93.0],
        "elec_in_ltw_%": [92.9, 94.3, 72.9, 77.1, 94.2, 96.7, 76.7, 84.7],
        "pv_wind_caputre_rate_%": [56.1, 55.5, 56.1, 55.5, 58.1, 59.4, 58.1, 59.4]
    },
    index=[
        "delayed w6",
        "delayed w6 wAllowed24h",
        "delayed w6 no flh",
        "delayed w6 no flh wAllowed24h",
        "synchronized w6",
        "synchronized w6 wAllowed24h",
        "synchronized w6 no flh",
        "synchronized w6 no flh wAllowed24h",
    ]
)
table2["is_synced"] = table2.index.str.contains("synchronized")
table2["is_synced"] = table2["is_synced"].map({False: "delayed", True: "synchronized"})
table2

,flex_elec_gwh,costs_savings_keur,emissions_saved_tco2,flh_w_flx_h,daily_min_capture_%,elec_in_ltw_%,pv_wind_caputre_rate_%,is_synced
delayed w6,12.0,489,632,7008,66.0,92.9,56.1,delayed
delayed w6 wAllowed24h,12.6,520,617,7008,81.0,94.3,55.5,delayed
delayed w6 no flh,16.4,621,842,3711,66.0,72.9,56.1,delayed
delayed w6 no flh wAllowed24h,16.4,632,816,3657,81.0,77.1,55.5,delayed
synchronized w6,12.7,518,670,7008,71.1,94.2,58.1,synchronized
synchronized w6 wAllowed24h,14.1,573,650,7008,93.0,96.7,59.4,synchronized
synchronized w6 no flh,16.6,633,846,3668,71.1,76.7,58.1,synchronized
synchronized w6 no flh wAllowed24h,17.0,658,811,3570,93.0,84.7,59.4,synchronized


In [4]:
timescale_uri = os.getenv("NOWUM_TIMESCALE_URI")
forest_db_uri = os.getenv("FOREST_DB_URI")


In [5]:
sql = """
select
	index as timestamp,
	sum(solar) / 1e3 as solar,
	sum(wind_offshore) / 1e3 as wind_offshore,
	sum(wind_onshore) / 1e3 as wind_onshore,
	(sum(solar) + sum(wind_offshore) + sum(wind_onshore)) / 1e3 as total
from entsoe.query_wind_and_solar_forecast
where country in ('DE_TRANSNET', 'DE_50HZ', 'DE_AMPRION', 'DE_TENNET')
and index between '2023-12-30' and '2025-01-01'
group by timestamp
order by timestamp asc
"""
ee_gen = pd.read_sql(sql, timescale_uri)
ee_gen["wind"] = ee_gen["wind_offshore"] + ee_gen["wind_onshore"]
ee_gen

,timestamp,solar,wind_offshore,wind_onshore,total,wind
0,2023-12-30 00:00:00+00:00,0.0,23.080,137.524,160.604,160.604
1,2023-12-30 00:15:00+00:00,0.0,23.088,138.048,161.136,161.136
2,2023-12-30 00:30:00+00:00,0.0,23.084,138.624,161.708,161.708
3,2023-12-30 00:45:00+00:00,0.0,23.084,139.212,162.296,162.296
4,2023-12-30 01:00:00+00:00,0.0,23.028,139.748,162.776,162.776
...,...,...,...,...,...,...
35324,2024-12-31 23:00:00+00:00,0.0,3.742,38.168,41.910,41.910
35325,2024-12-31 23:15:00+00:00,0.0,3.756,38.390,42.146,42.146
35326,2024-12-31 23:30:00+00:00,0.0,3.764,38.584,42.348,42.348
35327,2024-12-31 23:45:00+00:00,0.0,3.776,38.747,42.523,42.523


In [6]:
optimizations = pd.read_sql("select distinct(optimization_case_name) as optis from flexible_power", forest_db_uri)["optis"].tolist()
power_timeseries = {}

for opti in tqdm(optimizations):
    if opti not in power_timeseries.keys():
        sql = f"select * from flexible_power where optimization_case_name = '{opti}' order by timestamp asc"
        power_timeseries[opti] = pd.read_sql(sql, forest_db_uri)

list(power_timeseries.keys())

  0%|          | 0/9 [00:00<?, ?it/s]

['dynamic_w6',
 'dynamic_w6_no_flh',
 'dynamic_w6_no_flh_wAllowed24h',
 'dynamic_w6_wAllowed24h',
 'predictive_w6',
 'predictive_w6_no_flh',
 'predictive_w6_no_flh_wAllowed24h',
 'predictive_w6_wAllowed24h',
 'static']

In [7]:
sql = """
    SELECT *
    FROM smard_from_2018.prices
    WHERE timestamp >= '2023-12-20 23:00'
    AND timestamp <= '2024-12-31 23:00'
    ORDER BY timestamp ASC
"""

smard = pd.read_sql(sql, timescale_uri)
smard

,timestamp,commodity_id,price
0,2023-12-20 23:00:00,4169,30.16
1,2023-12-20 23:15:00,4169,30.16
2,2023-12-20 23:30:00,4169,30.16
3,2023-12-20 23:45:00,4169,30.16
4,2023-12-21 00:00:00,4169,25.08
...,...,...,...
36188,2024-12-31 22:00:00,4169,0.52
36189,2024-12-31 22:15:00,4169,0.52
36190,2024-12-31 22:30:00,4169,0.52
36191,2024-12-31 22:45:00,4169,0.52


In [8]:
def slice_df(df, start, end):
    return df[(df["timestamp"] >= start) & (df["timestamp"] <= end)].reset_index(drop=True)

In [9]:
def get_windows(df):
    windows = []
    waiting_for_end = False

    for idx, row in df.sort_values("timestamp").iterrows():
        if not waiting_for_end:
            if row["low_price_window"] == 1:
                win_start = row["timestamp"]
                waiting_for_end = True
                continue

        if waiting_for_end:
            if row["low_price_window"] != 1:
                win_end = row["timestamp"]
                waiting_for_end = False

                windows.append((win_start, win_end))
    return windows

In [10]:
def build_figure(ee_gen, power_ts, smard, start, end, fname=None):

    ee_slc = slice_df(ee_gen, start, end)
    power_ts_slc = slice_df(power_ts, start, end)
    smard_slc = slice_df(smard, start, end)
    windows = get_windows(power_ts_slc)

    fig = make_subplots(rows=2, cols=1, specs=[[{"secondary_y": False}], [{"secondary_y": True}]])
    fig.add_trace(go.Scatter(x=ee_slc["timestamp"], y=ee_slc["wind"], name="Wind onshore + offshore"), row=1, col=1)
    fig.add_trace(go.Scatter(x=ee_slc["timestamp"], y=ee_slc["solar"], name="Photovoltaic"), row=1, col=1)
    fig.add_trace(go.Scatter(x=power_ts_slc["timestamp"], y=power_ts_slc["electricity_used"] / 1e3, name="Flexible asset utilization", line={"color": "#3BADEF"}), row=2, col=1)
    fig.add_trace(go.Scatter(x=smard_slc["timestamp"], y=smard_slc["price"], name="Electricity market price"), row=2, col=1, secondary_y=True)
    fig.update_yaxes(title="GW", row=1, secondary_y=False)
    fig.update_yaxes(title="MW", row=2, secondary_y=False)
    fig.update_yaxes(title="€ / MWh", row=2, secondary_y=True)
    fig.update_layout(legend=dict(orientation="h"))

    for win_start, win_end in windows:
        fig.add_vrect(win_start, win_end, opacity=0.25, fillcolor="purple", line_width=0)

    if fname is not None:
        fig.write_image(f"./figures/{fname}")
        time.sleep(3)
        fig.write_image(f"./figures/{fname}")

    return fig

In [21]:
build_figure(ee_gen, power_timeseries["dynamic_w6"], smard, '2024-06-05', '2024-06-11', "fig_dynamic_w6_summer_week.pdf")

In [20]:
build_figure(ee_gen, power_timeseries["predictive_w6_wAllowed24h"], smard, '2024-06-05', '2024-06-11', "fig_predictive_w6_24h_summer_week.pdf")

In [19]:
build_figure(ee_gen, power_timeseries["predictive_w6_no_flh_wAllowed24h"], smard, '2024-06-02', '2024-06-09', "fig_predictive_w6_no_flh_24h.pdf")